# Setup & Initialization

In [1]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
from datetime import datetime
import ConnectionConfig as cc
from importlib import reload

from pyspark.sql.functions import *

In [2]:
reload(cc)

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

/home/jovyan/work/02-treasure-hunt-dwh/ConnectionConfig.py
Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/s

In [3]:
spark = cc.startLocalCluster("dimUserInit")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


## Reading the source table

In [4]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

jdbc:postgresql://postgres-data4:5432/catchem?user=postgres&password=data&ssl=false


In [5]:
# Load DataFrames
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_treasure_log = cc.read_postgres_by_date(spark, "treasure_log", "log_time")
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_country = cc.read_postgres_without_partition(spark, "country")

print('user size: ', df_operational_user.count()) ## 486523
print('treasure size: ', df_operational_treasure.count()) ## 128294
print('treasure_log size: ', df_operational_treasure_log.count()) ## 1789438
print('city size: ', df_operational_city.count()) ## 1264588
print('country size: ', df_operational_country.count()) ## 249

user size:  486523
treasure size:  128294
treasure_log size:  1789439
city size:  1264588
country size:  249


# Transforming the source to the dimension format

### Selecting and renaming columns for easier joins and transformations; also, we can filter or aggregate here if needed to optimize later steps

In [6]:
df_operational_city = df_operational_city.select(
    "city_id",
    "city_name",
    F.col("country_code").alias("country_id")
)

df_operational_country = df_operational_country.select(
    F.col("code").alias("country_id"),
    F.col("name").alias("country_name")
)

df_operational_user = df_operational_user.select(
    F.col("id").alias("user_id"),
    F.col("city_city_id").alias("city_id")
)

df_operational_treasure_log = df_operational_treasure_log \
    .groupBy(F.col("hunter_id").alias("user_id")) \
    .agg(F.count("*").alias("treasure_found"))

df_operational_treasure = df_operational_treasure \
    .select(F.col("owner_id").alias("user_id")) \
    .distinct()

print('user size: ', df_operational_user.count()) ## 486523
print('treasure size: ', df_operational_treasure.count()) ## 33830
print('treasure_log size: ', df_operational_treasure_log.count()) ## 486340
print('city size: ', df_operational_city.count()) ## 1264588
print('country size: ', df_operational_country.count()) ## 249

user size:  486523
treasure size:  33830
treasure_log size:  486340
city size:  1264588
country size:  249


In [10]:
# Join location data (Inner join usually fine for City -> Country)
df_location = df_operational_city.join(df_operational_country, "country_id")

# Prepare the "Dedicator" flag (Avoid distinct here to save performance; join handles it)
df_is_dedicator = df_operational_treasure.withColumn("dedicator", F.lit(True))

# Main Transformation
df_transformed = df_operational_user \
    .join(df_location, "city_id", "left") \
    .join(df_operational_treasure_log, "user_id", "left") \
    .join(df_is_dedicator, "user_id", "left") \
    .select(
    "user_id",
    "city_name",
    "country_name",
    # Use coalesce to handle nulls for users with no logs/flags
    F.coalesce(F.col("treasure_found"), F.lit(0)).cast("integer").alias("treasure_found"),
    F.coalesce(F.col("dedicator"), F.lit(False)).alias("dedicator")
) \
    .withColumn("experience_level",
                F.when(F.col("treasure_found") == 0, "Starter")
                .when(F.col("treasure_found") < 4, "Amateur")
                .when(F.col("treasure_found") <= 10, "Professional")
                .otherwise("Pirate")
                ) \
    .withColumn("return_type",
                F.when(F.col("treasure_found") == 0, "Starter")
                .when(F.col("treasure_found") == 1 , "Not Returned")
                .when(F.col("treasure_found") > 1, "Returned")
                .otherwise("Something Went Wrong")
                ) \
    .withColumn("scd_start", F.current_timestamp()) \
    .withColumn("scd_end", F.lit(None).cast("timestamp")) \
    .withColumn("current", F.lit(True))

df_transformed = df_transformed.drop("treasure_found")

df_transformed.show(10, truncate=False)

+-------------------------------------------------+-------------------+------------------------+---------+----------------+------------+--------------------------+-------+-------+
|user_id                                          |city_name          |country_name            |dedicator|experience_level|return_type |scd_start                 |scd_end|current|
+-------------------------------------------------+-------------------+------------------------+---------+----------------+------------+--------------------------+-------+-------+
|[00 02 66 4B E2 A6 4D 31 BE C6 3C 26 2B DA 05 B0]|Porbandar Mill Pura|India                   |false    |Amateur         |Returned    |2026-06-11 17:59:07.216492|NULL   |true   |
|[00 05 0C 95 84 B6 45 82 80 D1 7F DC 34 C4 75 60]|Kinhupur           |India                   |false    |Amateur         |Returned    |2026-06-11 17:59:07.216492|NULL   |true   |
|[00 07 66 C4 1B 47 49 59 98 B9 58 A9 66 C2 7A 8A]|Melaka             |Malaysia                |true

In [11]:
df_transformed.groupBy("return_type").count().show()

+------------+------+
| return_type| count|
+------------+------+
|    Returned|464619|
|Not Returned| 21721|
|     Starter|   183|
+------------+------+



In [12]:
df_transformed.groupBy("experience_level").count().show()

+----------------+------+
|experience_level| count|
+----------------+------+
|         Amateur|430951|
|          Pirate| 33389|
|    Professional| 22000|
|         Starter|   183|
+----------------+------+



In [10]:
# S1 - Q6 How many users have created treasures vs how many haven't?
from pyspark.sql.window import Window

# 1. Group and count
df_counts = df_transformed.groupBy("dedicator").count()

# 2. Define a Window to get the total sum
window_spec = Window.partitionBy()

# 3. Calculate percentage
df_counts.withColumn("percentage",
    (F.col("count") / F.sum("count").over(window_spec)) * 100
).show()

+---------+------+-----------------+
|dedicator| count|       percentage|
+---------+------+-----------------+
|    false|452693|93.04657744854816|
|     true| 33830|6.953422551451832|
+---------+------+-----------------+



In [23]:
# S2 - Q3. Do users who created treasures themselves more often find treasures then users who don’t?

# Group by dedicator and look at the distribution of experience levels
df_analysis = df_transformed.filter(F.col("experience_level") != "Starter").groupBy("experience_level", "dedicator").count()

# 2. Define the Window partitioned by experience_level
# This ensures the percentages sum to 100% for EACH level (Amateur, Pirate, etc.)
window_by_level = Window.partitionBy("experience_level")

# 3. Calculate the percentage
df_result = df_analysis.withColumn("pct_of_level",
    F.round((F.col("count") / F.sum("count").over(window_by_level)) * 100, 2)
)

# 4. Show the results ordered by level
df_result.orderBy("experience_level", "dedicator").show()

+----------------+---------+------+------------+
|experience_level|dedicator| count|pct_of_level|
+----------------+---------+------+------------+
|         Amateur|    false|430923|       99.99|
|         Amateur|     true|    29|        0.01|
|          Pirate|     true| 33389|       100.0|
|    Professional|    false| 21587|       98.13|
|    Professional|     true|   412|        1.87|
+----------------+---------+------+------------+



In [ ]:
# 1. We need to re-attach the count of finds since you dropped it earlier
# If you don't have the raw count, we can derive it from the experience_level
# or better yet, run this before you drop the 'treasure_found' column.

activity_analysis = df_transformed.groupBy("dedicator").agg(
    F.count("user_id").alias("total_user_count"),
    # Calculate the percentage of users in this group who are NOT 'Starters'
    F.round(
        (F.sum(F.when(F.col("experience_level") != "Starter", 1).otherwise(0)) /
         F.count("user_id")) * 100, 2
    ).alias("active_hunter_rate_pct")
)

activity_analysis.show()

## Deleting the dimension table if it already exists

In [8]:
cc.delete_delta_table(spark, "dim_user")

Successfully deleted metadata and files for dim_user


# Creating the dimension table

In [9]:
from pyspark.sql.types import LongType, StringType, TimestampType, IntegerType, BooleanType, BinaryType

#Initialize the table with the identity column

DeltaTable.create(spark) \
    .tableName("dim_user") \
    .addColumn("user_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("user_id", BinaryType(), nullable=False) \
    .addColumn("city_name", StringType()) \
    .addColumn("country_name", StringType()) \
    .addColumn("experience_level", StringType()) \
    .addColumn("return_type", StringType()) \
    .addColumn("dedicator", BooleanType()) \
    .addColumn("scd_start", TimestampType()) \
    .addColumn("scd_end", TimestampType()) \
    .addColumn("current", BooleanType()) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

# Populating the dimension table


In [10]:
#Write to the table (identity column will be auto-generated)
df_transformed.write.format("delta").mode("append").saveAsTable("dim_user")

## Save as Parquet format


In [5]:
# Delete parquet file if it already exists
cc.delete_parquet_table("parquet/dim_user_parquet")

Deleted old parquet at parquet/dim_user_parquet


In [6]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/dim_user")


# Cast the user_id column from Binary to String
df = df.withColumn("user_id", F.hex(F.col("user_id")))

# Add the new experience_level_id column mapping text to numbers
df = df.withColumn(
    "experience_level_id",
    F.when(F.col("experience_level") == "Starter", 1)
     .when(F.col("experience_level") == "Amateur", 2)
     .when(F.col("experience_level") == "Professional", 3)
     .when(F.col("experience_level") == "Pirate", 4)
     .otherwise(None) # Sets to Null if it's none of the above
)


df.coalesce(1).write.format("parquet").save("./parquet/dim_user_parquet")

## Checking the results

In [6]:
spark.read.format("delta").load("spark-warehouse/dim_user").select(F.count("*")).show(truncate=False)

+--------+
|count(1)|
+--------+
|486523  |
+--------+



In [7]:
import pandas as pd

parquet_path = "parquet/dim_user_parquet/"

# Load into a Pandas DataFrame
df_pandas = pd.read_parquet(parquet_path)

# Print the list of column names
print(df_pandas.columns.tolist())

['user_sk', 'user_id', 'city_name', 'country_name', 'experience_level', 'return_type', 'dedicator', 'scd_start', 'scd_end', 'current']


In [13]:
spark.read.format("delta").load("spark-warehouse/dim_user").filter(F.col("dedicator") == True).limit(10).show(
    truncate=False)

+-------+-------------------------------------------------+--------------------------+------------------+----------------+---------+--------------------------+-------+-------+
|user_sk|user_id                                          |city_name                 |country_name      |experience_level|dedicator|scd_start                 |scd_end|current|
+-------+-------------------------------------------------+--------------------------+------------------+----------------+---------+--------------------------+-------+-------+
|99     |[00 09 31 D5 92 34 49 F7 AA 89 0D D6 92 60 6B B6]|Chandagala                |India             |Pirate          |true     |2026-03-19 15:00:32.566717|NULL   |true   |
|123    |[00 0D A8 C4 C4 88 49 70 A4 5D 4E 9C E5 93 10 1F]|Shimohideya Nishi         |Japan             |Pirate          |true     |2026-03-19 15:00:32.566717|NULL   |true   |
|251    |[00 1F F2 26 01 22 4F D4 94 08 54 10 C4 58 20 E1]|Gannavaram Centre         |India             |Pirate         

In [30]:
spark.sql("SELECT * FROM dim_user WHERE dedicator = true LIMIT 10").show(truncate=False)

+-------+-------------------------------------------------+----------+-------------+------------------+------------------------+----------------+---------+--------------+--------------------------+-------+-------+
|user_sk|user_id                                          |first_name|last_name    |city_name         |country_name            |Experience_level|dedicator|treasure_found|scd_start                 |scd_end|current|
+-------+-------------------------------------------------+----------+-------------+------------------+------------------------+----------------+---------+--------------+--------------------------+-------+-------+
|8      |[00 07 66 C4 1B 47 49 59 98 B9 58 A9 66 C2 7A 8A]|Alexanne  |Cronin       |Melaka            |Malaysia                |Pirate          |true     |19            |2026-03-14 22:05:33.485977|NULL   |true   |
|44     |[00 0B 62 1C D5 70 4A 17 AD 2A 6B 2F B8 0B 09 6D]|Elmira    |Quigley      |Maysville         |United States of America|Pirate          

## Cleaning up the environment (stop Spark and reset variables)

In [7]:
spark.stop()

In [4]:
%reset